# US Real Estate Affordability

Home prices on their own say nothing about affordability — a \$400K house means very
different things in Cleveland versus San Jose. The right lens is the price-to-income
ratio: median home value divided by local median income. By international convention,
anything above 5x is considered unaffordable.

This works through Zillow's home value index for ~500 large cities, joins state median
incomes from ACS 2019, and ranks where housing has come unmoored from what people earn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

!kaggle datasets download -d paultimothymooney/zillow-house-price-data --unzip -q

# ZHVI = Zillow Home Value Index, smoothed typical home value
zhvi = pd.read_csv('City_Zhvi_AllHomes.csv', encoding='latin-1')
zhvi.shape

## Shaping the data

The ZHVI file is wide — one column per month. Take the latest available month (Dec 2019)
as the home value, keep the city identifiers, and drop the rest.

In [ ]:
price_col = '2019-12-31'
homes = zhvi[['RegionName', 'State', 'Metro', 'SizeRank', price_col]].copy()
homes.columns = ['city', 'state', 'metro', 'size_rank', 'home_value']
homes = homes.dropna(subset=['home_value', 'state'])

# ACS 2019 median household income by state
income = {
    'AL':51734,'AK':77640,'AZ':62055,'AR':47062,'CA':80440,'CO':77127,'CT':78833,
    'DE':70176,'FL':59227,'GA':61980,'HI':83102,'ID':60999,'IL':69187,'IN':57603,
    'IA':61836,'KS':59597,'KY':52238,'LA':51073,'ME':58924,'MD':87063,'MA':85843,
    'MI':59584,'MN':74593,'MS':45081,'MO':57409,'MT':56539,'NE':63229,'NV':63276,
    'NH':77933,'NJ':85751,'NM':51945,'NY':72108,'NC':57341,'ND':64894,'OH':58116,
    'OK':53840,'OR':67058,'PA':63463,'RI':70305,'SC':56227,'SD':59533,'TN':56071,
    'TX':64034,'UT':74197,'VT':63001,'VA':76456,'WA':78687,'WV':48037,'WI':63293,
    'WY':65304,'DC':92266,
}
homes = homes[homes['state'].isin(income)]
homes['state_income'] = homes['state'].map(income)
homes['pti'] = homes['home_value'] / homes['state_income']

Focus on the largest cities so the ranking is about places people have heard of and
where the data is most reliable — top 500 by Zillow's size rank.

In [ ]:
df = homes[homes['size_rank'] <= 500].copy()

REGION = {
    **dict.fromkeys(['CA','OR','WA','NV','AZ','UT','CO','ID','MT','WY','AK','HI'], 'West'),
    **dict.fromkeys(['TX','FL','GA','NC','SC','VA','TN','AL','MS','AR','LA','OK','KY',
                     'WV','MD','DE','DC','NM'], 'South'),
    **dict.fromkeys(['NY','NJ','PA','CT','MA','RI','NH','VT','ME'], 'Northeast'),
    **dict.fromkeys(['IL','OH','MI','IN','WI','MN','IA','MO','KS','NE','SD','ND'],
                    'Midwest'),
}
df['region'] = df['state'].map(REGION)
print(f"{len(df)} cities")
df['pti'].describe().round(2)

## How widespread is unaffordability?

The 5x line is the standard cutoff. Count how many of these large cities clear it.

In [ ]:
unaffordable = (df['pti'] > 5).mean()
print(f"{unaffordable:.0%} of cities sit above the 5x unaffordable line")
print(f"National median PTI: {df['pti'].median():.1f}x")

print("\nLeast affordable:")
print(df.nlargest(5, 'pti')[['city', 'state', 'home_value', 'pti']]
        .to_string(index=False))

## Regional split

Average PTI by region. The West/Midwest spread is the story — same country, very different
relationship between housing and earnings.

In [ ]:
df.groupby('region')['pti'].agg(['mean', 'median', 'count']).round(2) \
  .sort_values('mean', ascending=False)

## Charts

In [ ]:
BG = '#0D1117'
GREEN, AMBER, RED, CORAL = '#00FF9D', '#FFB830', '#E84545', '#FF6B5E'

def tier(p):
    if p > 8: return 'Severe (>8x)'
    if p > 5: return 'Unaffordable (5-8x)'
    if p > 4: return 'Moderate (4-5x)'
    return 'Affordable (<4x)'
df['tier'] = df['pti'].apply(tier)
TIER_COLOR = {'Affordable (<4x)': GREEN, 'Moderate (4-5x)': AMBER,
              'Unaffordable (5-8x)': CORAL, 'Severe (>8x)': RED}

def style_ax(ax):
    ax.set_facecolor(BG)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    for s in ['bottom', 'left']: ax.spines[s].set_color('#30363D')
    ax.tick_params(colors='#8B949E')

In [ ]:
# Chart 1 — tier split, regional averages, headline numbers
fig, axes = plt.subplots(1, 3, figsize=(18, 7)); fig.patch.set_facecolor(BG)
order = ['Affordable (<4x)', 'Moderate (4-5x)', 'Unaffordable (5-8x)', 'Severe (>8x)']
counts = df['tier'].value_counts().reindex(order)

axes[0].pie(counts, colors=[TIER_COLOR[t] for t in order], autopct='%1.0f%%',
            startangle=90, wedgeprops=dict(width=0.55, edgecolor=BG, linewidth=3),
            textprops=dict(color='white', fontsize=8))
axes[0].legend(order, loc='lower center', bbox_to_anchor=(0.5, -0.18),
               facecolor='#161B22', labelcolor='white', edgecolor='#30363D', fontsize=7)
axes[0].set_title('Affordability tiers', color='white', fontweight='bold', pad=12)

ax = axes[1]; style_ax(ax)
reg = df.groupby('region')['pti'].mean().reindex(['West', 'Northeast', 'South', 'Midwest'])
ax.bar(reg.index, reg.values, color=[RED, CORAL, AMBER, GREEN], width=0.55)
ax.axhline(5, color='white', ls='--', alpha=0.4)
ax.set_ylabel('Avg price-to-income', color='#8B949E')
ax.set_title('Affordability by region', color='white', fontweight='bold', pad=12)
for i, v in enumerate(reg.values):
    ax.text(i, v + 0.1, f'{v:.1f}x', ha='center', color='white', fontweight='bold')
for t in ax.get_xticklabels(): t.set_fontsize(8)

ax = axes[2]; ax.axis('off'); ax.set_facecolor('#161B22')
worst = df.nlargest(1, 'pti').iloc[0]
gap = reg['West'] / reg['Midwest']
notes = [(f'{unaffordable:.0%}', 'cities above 5x', RED),
         (f"{worst['pti']:.1f}x", f"highest: {worst['city']}, {worst['state']}", CORAL),
         (f"{df['pti'].median():.1f}x", 'national median', AMBER),
         (f'{gap:.1f}x', 'West vs Midwest gap', GREEN)]
for i, (num, lbl, c) in enumerate(notes):
    y = 0.82 - i * 0.22
    ax.text(0.5, y, num, ha='center', fontsize=20, fontweight='bold', color=c)
    ax.text(0.5, y - 0.08, lbl, ha='center', fontsize=8, color='#8B949E')
ax.set_title('Headline numbers', color='white', fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig('chart_01_affordability_overview.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# Chart 2 — least and most affordable cities
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8)); fig.patch.set_facecolor(BG)

style_ax(ax1)
worst = df.nlargest(15, 'pti').sort_values('pti')
ax1.barh(worst['city'] + ', ' + worst['state'], worst['pti'],
         color=[RED if p > 8 else CORAL for p in worst['pti']], height=0.65)
for i, (_, r) in enumerate(worst.iterrows()):
    ax1.text(r['pti'] + 0.1, i, f"{r['pti']:.1f}x · ${r['home_value']/1e6:.1f}M",
             va='center', fontsize=8, color='#8B949E')
ax1.axvline(5, color='white', ls='--', alpha=0.4)
ax1.set_xlabel('Price-to-income', color='#8B949E')
ax1.set_title('15 least affordable cities', color='white', fontweight='bold', pad=12)

style_ax(ax2)
best = df[df['home_value'] > 50000].nsmallest(15, 'pti').sort_values('pti',
                                                                     ascending=False)
ax2.barh(best['city'] + ', ' + best['state'], best['pti'], color=GREEN, height=0.65)
for i, (_, r) in enumerate(best.iterrows()):
    ax2.text(r['pti'] + 0.02, i, f"{r['pti']:.1f}x · ${r['home_value']/1e3:.0f}K",
             va='center', fontsize=8, color='#8B949E')
ax2.set_xlabel('Price-to-income', color='#8B949E')
ax2.set_title('15 most affordable cities', color='white', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('chart_02_best_worst_cities.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# Chart 3 — value vs income with affordability lines, and PTI spread by region
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8)); fig.patch.set_facecolor(BG)
RC = {'West': RED, 'Northeast': '#FF9B30', 'South': '#00D4FF', 'Midwest': GREEN}

style_ax(ax1)
for region, c in RC.items():
    sub = df[df['region'] == region].sample(
        min(80, (df.region == region).sum()), random_state=42)
    ax1.scatter(sub['state_income'] / 1000, sub['home_value'] / 1000,
                c=c, s=28, alpha=0.6, label=region)
xs = np.linspace(40, 100, 50)
for mult, ls in [(4, '--'), (5, '-'), (8, ':')]:
    ax1.plot(xs, xs * mult, color='white', ls=ls, alpha=0.3,
             label=f'{mult}x line')
ax1.set_xlabel('State median income ($K)', color='#8B949E')
ax1.set_ylabel('Home value ($K)', color='#8B949E')
ax1.set_title('Home value vs income', color='white', fontweight='bold', pad=12)
ax1.legend(facecolor='#161B22', labelcolor='white', edgecolor='#30363D',
           ncol=2, fontsize=8)

style_ax(ax2)
regions = ['West', 'Northeast', 'South', 'Midwest']
bp = ax2.boxplot([df[df['region'] == r]['pti'] for r in regions],
                 patch_artist=True, labels=regions)
for patch, r in zip(bp['boxes'], regions):
    patch.set_facecolor(RC[r]); patch.set_alpha(0.35)
for med in bp['medians']: med.set_color('white'); med.set_linewidth(2)
ax2.axhline(5, color='white', ls='--', alpha=0.4)
ax2.set_ylabel('Price-to-income', color='#8B949E')
ax2.set_title('PTI distribution by region', color='white', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('chart_03_scatter_boxplot.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# Chart 4 — market classification and the state-level ranking
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 9)); fig.patch.set_facecolor(BG)

def market(row):
    if row['pti'] < 3 and row['home_value'] > 100000: return ('Buy opportunity', GREEN)
    if row['pti'] > 8: return ('Severely overvalued', RED)
    if row['pti'] > 5: return ('Overvalued', CORAL)
    if row['pti'] < 4: return ('Fair value', '#00D4FF')
    return ('Moderately overvalued', AMBER)
mk = df.apply(market, axis=1)
df['market'] = [m[0] for m in mk]

style_ax(ax1)
for cls in df['market'].unique():
    sub = df[df['market'] == cls]
    color = market(sub.iloc[0])[1]
    ax1.scatter(sub['pti'], sub['home_value'] / 1000, c=color,
                s=40, alpha=0.75, label=f'{cls} ({len(sub)})')
ax1.axvline(5, color='#30363D', ls='--', alpha=0.6)
ax1.set_xlabel('Price-to-income', color='#8B949E')
ax1.set_ylabel('Home value ($K)', color='#8B949E')
ax1.set_title('Market classification', color='white', fontweight='bold', pad=12)
ax1.legend(facecolor='#161B22', labelcolor='white', edgecolor='#30363D',
           fontsize=8, loc='upper left')

style_ax(ax2)
state_pti = df.groupby('state')['pti'].mean().sort_values()
ends = pd.concat([state_pti.head(10), state_pti.tail(10)])
colors = [GREEN if v < 3.5 else AMBER if v < 5 else RED for v in ends.values]
ax2.barh(ends.index, ends.values, color=colors, height=0.65)
for i, v in enumerate(ends.values):
    ax2.text(v + 0.05, i, f'{v:.1f}x', va='center', color='white',
             fontweight='bold', fontsize=8)
ax2.axvline(5, color='white', ls='--', alpha=0.35)
ax2.set_xlabel('Avg price-to-income', color='#8B949E')
ax2.set_title('Most and least affordable states', color='white', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('chart_04_investment_matrix.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## Takeaways

- About a third of large US cities are above the 5x unaffordability line — the problem has
  spread well beyond the obvious coastal metros.
- The West (~6.8x avg) is more than twice as stretched as the Midwest (~2.8x).
- California dominates the most-unaffordable ranking, with Santa Monica around 20x — a
  number that has no relationship to local incomes.
- The most affordable markets cluster in Ohio, the broader Midwest, and the South, where
  prices and incomes are still roughly in line — the clearest "buy" signals on an
  income-adjusted basis.